# Isobutyl Alcohol

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
secbutylalcohol,74.122,3.3102,3.34415596,222.7977705,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
secbutylalcohol,H,secbutylalcohol,e,2000.0,0.03
"""

model = PCSAFT(["secbutylalcohol"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_secbutylalcohol.csv")
fix_line_endings("rhol_secbutylalcohol.csv")

Fixed: satp_secbutylalcohol.csv
Fixed: rhol_secbutylalcohol.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [17]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 3.2,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 3.2, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.001)

In [18]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_secbutylalcohol.csv",
        "satp_secbutylalcohol.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 1.4233212281605299


In [19]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([3.1672553031234445, 217.38153645546154, 3.387137202754401, 2556.948926631885, 0.01103159197093655], PCSAFT{BasicIdeal, Float64}("secbutylalcohol"))

In [20]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [21]:
aard_p   = calculate_AAD(model_opt, "satp_secbutylalcohol.csv",   my_saturation_p)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: satp_secbutylalcohol.csv ===
Clapeyron Estimator  exp           calc          ARD%    
280.7300    662.000000    659.227546    0.4188  
281.0500    675.000000    675.916960    0.1358  
284.4800    878.000000    879.880217    0.2141  
287.9100    1136.000000   1136.706747   0.0622  
292.1600    1542.000000   1545.454863   0.2241  
292.2600    1552.000000   1556.460834   0.2874  
294.4500    1812.000000   1815.463041   0.1911  
298.0900    2347.000000   2330.392194   0.7076  
301.0200    2847.000000   2833.806867   0.4634  
303.9000    3412.000000   3418.897644   0.2022  
308.1500    4467.000000   4474.257768   0.1625  
311.6000    5515.000000   5528.903276   0.2521  
314.4100    6551.000000   6540.883196   0.1544  
AARD = 0.2674%


0.26736824702650835

In [22]:
aard_p   = calculate_AAD(model_opt, "rhol_secbutylalcohol.csv",   my_liquid_density)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: rhol_secbutylalcohol.csv ===
Clapeyron Estimator  exp           calc          ARD%    
293.1500    806.700000    806.854355    0.0191  
298.1500    802.700000    802.614135    0.0107  
303.1500    798.400000    798.338028    0.0078  
308.1500    794.100000    794.020838    0.0100  
313.1500    789.800000    789.657372    0.0181  
318.1500    785.200000    785.242425    0.0054  
323.1500    780.600000    780.770768    0.0219  
AARD = 0.0133%


0.013271465443189159